# 01c — Weather Data Preparation
Downloads and cleans NOAA ISD-Lite hourly weather data for 100 major US airports (2023–Aug 2025). Weather data availability through August 2025 defines the dataset cutoff date.
**Source:** [NOAA ISD-Lite](https://www.ncei.noaa.gov/products/land-based-station/integrated-surface-database)
**Output:** `dataset/weather/weather_clean.parquet` — 100 airports, 2.3M hourly observations

In [ ]:
import urllib.request
import os
import pandas as pd
import gzip

os.makedirs('dataset/weather/raw', exist_ok=True)

airport_stations = {
    'ATL': '722190-13874', 'AUS': '722540-13904', 'BNA': '723270-13897',
    'BOS': '725090-14739', 'BWI': '724060-93721', 'CLE': '725240-14820',
    'CLT': '723140-13881', 'CMH': '724280-14821', 'CVG': '724210-93814',
    'DAL': '722580-13960', 'DCA': '724050-13743', 'DEN': '725650-03017',
    'DFW': '722590-03927', 'DTW': '725370-94847', 'EWR': '725020-14734',
    'FLL': '747830-12849', 'HNL': '911820-22521', 'HOU': '722440-12918',
    'IAD': '724030-93738', 'IAH': '722430-12960', 'IND': '724380-93819',
    'JFK': '744860-94789', 'LAS': '723860-23169', 'LAX': '722950-23174',
    'LGA': '725030-14732', 'MCI': '724460-03947', 'MCO': '722050-12815',
    'MDW': '725340-14819', 'MIA': '722020-12839', 'MSP': '726580-14922',
    'MSY': '722310-12916', 'OAK': '724930-23230', 'ORD': '725300-94846',
    'PDX': '726980-24229', 'PHL': '724080-13739', 'PHX': '722780-23183',
    'PIT': '725200-94823', 'RDU': '723060-13722', 'RSW': '722108-12894',
    'SAN': '722900-23188', 'SAT': '722530-12921', 'SEA': '727930-24233',
    'SFO': '724940-23234', 'SJC': '724945-23293', 'SJU': '785260-11641',
    'SLC': '725720-24127', 'SMF': '724839-93225', 'SNA': '722977-93184',
    'STL': '724340-13994', 'TPA': '722110-12842',
}

base_url = 'https://www.ncei.noaa.gov/pub/data/noaa/isd-lite'
years = ['2023', '2024', '2025']

total = len(airport_stations) * len(years)
done = 0

for airport, station_id in airport_stations.items():
    for year in years:
        filename = f'{station_id}-{year}.gz'
        url = f'{base_url}/{year}/{filename}'
        save_path = f'dataset/weather/raw/{airport}_{year}.gz'
        
        try:
            urllib.request.urlretrieve(url, save_path)
            done += 1
            print(f'[{done}/{total}] Downloaded {airport} {year}')
        except Exception as e:
            print(f'[FAILED] {airport} {year}: {e}')

print(f'\nDone! {done}/{total} files downloaded.')

[1/150] Downloaded ATL 2023
[2/150] Downloaded ATL 2024
[3/150] Downloaded ATL 2025
[4/150] Downloaded AUS 2023
[5/150] Downloaded AUS 2024
[6/150] Downloaded AUS 2025
[7/150] Downloaded BNA 2023
[8/150] Downloaded BNA 2024
[9/150] Downloaded BNA 2025
[10/150] Downloaded BOS 2023
[11/150] Downloaded BOS 2024
[12/150] Downloaded BOS 2025
[13/150] Downloaded BWI 2023
[14/150] Downloaded BWI 2024
[15/150] Downloaded BWI 2025
[16/150] Downloaded CLE 2023
[17/150] Downloaded CLE 2024
[18/150] Downloaded CLE 2025
[19/150] Downloaded CLT 2023
[20/150] Downloaded CLT 2024
[21/150] Downloaded CLT 2025
[22/150] Downloaded CMH 2023
[23/150] Downloaded CMH 2024
[24/150] Downloaded CMH 2025
[25/150] Downloaded CVG 2023
[26/150] Downloaded CVG 2024
[27/150] Downloaded CVG 2025
[28/150] Downloaded DAL 2023
[29/150] Downloaded DAL 2024
[30/150] Downloaded DAL 2025
[31/150] Downloaded DCA 2023
[32/150] Downloaded DCA 2024
[33/150] Downloaded DCA 2025
[34/150] Downloaded DEN 2023
[35/150] Downloaded DEN

In [ ]:
col_names = ['YEAR', 'MONTH', 'DAY', 'HOUR', 'TEMP', 'DEW_POINT', 
             'PRESSURE', 'WIND_DIR', 'WIND_SPEED', 'SKY_CONDITION',
             'PRECIP_1HR', 'PRECIP_6HR']

all_weather = []
raw_dir = 'dataset/weather/raw'

for f in sorted(os.listdir(raw_dir)):
    if not f.endswith('.gz'):
        continue
    airport = f.split('_')[0]
    filepath = os.path.join(raw_dir, f)
    
    try:
        with gzip.open(filepath, 'rt') as gz:
            df = pd.read_fwf(gz, header=None, names=col_names)
        df['AIRPORT'] = airport
        all_weather.append(df)
        print(f'Parsed {f}: {len(df):,} rows')
    except Exception as e:
        print(f'FAILED {f}: {e}')

weather = pd.concat(all_weather, ignore_index=True)
print(f'\nTotal: {len(weather):,} rows')
print(f'Airports: {weather["AIRPORT"].nunique()}')

Parsed ATL_2023.gz: 8,760 rows
Parsed ATL_2024.gz: 8,780 rows
Parsed ATL_2025.gz: 5,713 rows
Parsed AUS_2023.gz: 8,759 rows
Parsed AUS_2024.gz: 8,784 rows
Parsed AUS_2025.gz: 5,717 rows
Parsed BNA_2023.gz: 8,760 rows
Parsed BNA_2024.gz: 8,784 rows
Parsed BNA_2025.gz: 5,717 rows
Parsed BOS_2023.gz: 8,760 rows
Parsed BOS_2024.gz: 8,777 rows
Parsed BOS_2025.gz: 5,711 rows
Parsed BWI_2023.gz: 8,759 rows
Parsed BWI_2024.gz: 8,782 rows
Parsed BWI_2025.gz: 5,717 rows
Parsed CLE_2023.gz: 8,756 rows
Parsed CLE_2024.gz: 8,784 rows
Parsed CLE_2025.gz: 5,686 rows
Parsed CLT_2023.gz: 8,751 rows
Parsed CLT_2024.gz: 8,772 rows
Parsed CLT_2025.gz: 5,715 rows
Parsed CMH_2023.gz: 8,759 rows
Parsed CMH_2024.gz: 8,783 rows
Parsed CMH_2025.gz: 5,716 rows
Parsed CVG_2023.gz: 8,759 rows
Parsed CVG_2024.gz: 8,781 rows
Parsed CVG_2025.gz: 5,711 rows
Parsed DAL_2023.gz: 8,757 rows
Parsed DAL_2024.gz: 8,783 rows
Parsed DAL_2025.gz: 5,715 rows
Parsed DCA_2023.gz: 8,757 rows
Parsed DCA_2024.gz: 8,780 rows
Parsed D

In [3]:
print(weather[weather['YEAR']==2025].groupby('AIRPORT')['DAY'].max().head())

AIRPORT
ATL    31
AUS    31
BNA    31
BOS    31
BWI    31
Name: DAY, dtype: int64


In [4]:
weather = weather.replace(-9999, pd.NA)
weather['TEMP'] = weather['TEMP'] / 10
weather['DEW_POINT'] = weather['DEW_POINT'] / 10
weather['PRESSURE'] = weather['PRESSURE'] / 10
weather['WIND_SPEED'] = weather['WIND_SPEED'] / 10
weather['PRECIP_1HR'] = weather['PRECIP_1HR'] / 10
weather['PRECIP_6HR'] = weather['PRECIP_6HR'] / 10

weather['DATETIME'] = pd.to_datetime(
    weather[['YEAR','MONTH','DAY','HOUR']].rename(
        columns={'YEAR':'year','MONTH':'month','DAY':'day','HOUR':'hour'}
    )
)

print(f"Weather: {weather['DATETIME'].min()} to {weather['DATETIME'].max()}")

Weather: 2023-01-01 00:00:00 to 2025-08-27 09:00:00


In [5]:
print(f"Null counts:")
print(weather[['TEMP','WIND_SPEED','PRESSURE','PRECIP_1HR','SKY_CONDITION']].isna().sum())

Null counts:
TEMP               1339
WIND_SPEED         1435
PRESSURE          17445
PRECIP_1HR        31045
SKY_CONDITION    517561
dtype: int64


In [6]:
weather.to_parquet('dataset/weather/weather_clean.parquet', index=False)
print(f"Saved: {len(weather):,} rows")

Saved: 1,161,374 rows


In [7]:
df=pd.read_parquet("/Users/harshithnr/flight_delay_predictions/dataset/weather/weather_clean.parquet")

In [8]:
df.head()

,YEAR,MONTH,DAY,HOUR,TEMP,DEW_POINT,PRESSURE,WIND_DIR,WIND_SPEED,SKY_CONDITION,PRECIP_1HR,PRECIP_6HR,AIRPORT,DATETIME
0,2023,1,1,0,15.6,13.9,1016.6,200.0,2.1,6.0,NaN,0.0,ATL,2023-01-01 00:00:00
1,2023,1,1,1,14.4,13.3,1016.7,0.0,0.0,2.0,0.0,NaN,ATL,2023-01-01 01:00:00
2,2023,1,1,2,13.9,12.8,1016.9,210.0,2.6,0.0,0.0,NaN,ATL,2023-01-01 02:00:00
3,2023,1,1,3,13.3,12.8,1017.3,230.0,2.1,2.0,0.0,NaN,ATL,2023-01-01 03:00:00
4,2023,1,1,4,13.3,12.8,1017.4,250.0,1.5,2.0,0.0,NaN,ATL,2023-01-01 04:00:00


In [9]:
df.AIRPORT.value_counts()

AIRPORT
BNA    23261
IAH    23261
AUS    23260
BWI    23258
CMH    23258
HOU    23256
DAL    23255
OAK    23255
DCA    23254
DEN    23254
IND    23254
IAD    23253
ATL    23253
DFW    23252
CVG    23251
LGA    23251
BOS    23248
LAX    23247
MCI    23245
PIT    23244
MIA    23244
HNL    23243
MDW    23238
CLT    23238
JFK    23237
PDX    23237
SAT    23233
PHL    23232
SJC    23229
SAN    23228
CLE    23226
SEA    23225
RDU    23222
LAS    23221
STL    23219
SFO    23215
FLL    23214
TPA    23213
EWR    23208
PHX    23207
SLC    23198
ORD    23197
MCO    23196
MSY    23194
SJU    23188
SNA    23178
RSW    23176
SMF    23170
MSP    23148
DTW    23130
Name: count, dtype: int64

In [ ]:
airport_stations_next30 = {
    'ABQ': '723650-23050', 'ANC': '702730-26451', 'BDL': '725080-14740',
    'BHM': '722280-13876', 'BOI': '726810-24131', 'BUF': '725280-14733',
    'BUR': '722880-23152', 'CHS': '722080-13880', 'ELP': '722700-23044',
    'GEG': '727850-24157', 'GRR': '726350-94860', 'JAX': '722060-13889',
    'KOA': '911975-21510', 'LGB': '722970-23129', 'LIH': '911650-22536',
    'MEM': '723340-13893', 'MKE': '726400-14839', 'OGG': '911900-22516',
    'OKC': '723530-13967', 'OMA': '725500-14942', 'ONT': '747040-03102',
    'ORF': '723080-13737', 'PBI': '722030-12844', 'RIC': '724010-13740',
    'RNO': '724880-23185', 'SAV': '722070-03822', 'SDF': '724230-93821',
    'SRQ': '722115-12871', 'TUL': '723560-13968', 'TUS': '722740-23160',
}

base_url = 'https://www.ncei.noaa.gov/pub/data/noaa/isd-lite'
years = ['2023', '2024', '2025']

total = len(airport_stations_next30) * len(years)
done = 0

for airport, station_id in airport_stations_next30.items():
    for year in years:
        filename = f'{station_id}-{year}.gz'
        url = f'{base_url}/{year}/{filename}'
        save_path = f'dataset/weather/raw/{airport}_{year}.gz'
        
        try:
            urllib.request.urlretrieve(url, save_path)
            done += 1
            print(f'[{done}/{total}] Downloaded {airport} {year}')
        except Exception as e:
            print(f'[FAILED] {airport} {year}: {e}')

print(f'\nDone! {done}/{total} files downloaded.')

[1/90] Downloaded ABQ 2023
[2/90] Downloaded ABQ 2024
[3/90] Downloaded ABQ 2025
[4/90] Downloaded ANC 2023
[5/90] Downloaded ANC 2024
[6/90] Downloaded ANC 2025
[7/90] Downloaded BDL 2023
[8/90] Downloaded BDL 2024
[9/90] Downloaded BDL 2025
[10/90] Downloaded BHM 2023
[11/90] Downloaded BHM 2024
[12/90] Downloaded BHM 2025
[13/90] Downloaded BOI 2023
[14/90] Downloaded BOI 2024
[15/90] Downloaded BOI 2025
[16/90] Downloaded BUF 2023
[17/90] Downloaded BUF 2024
[18/90] Downloaded BUF 2025
[19/90] Downloaded BUR 2023
[20/90] Downloaded BUR 2024
[21/90] Downloaded BUR 2025
[22/90] Downloaded CHS 2023
[23/90] Downloaded CHS 2024
[24/90] Downloaded CHS 2025
[25/90] Downloaded ELP 2023
[26/90] Downloaded ELP 2024
[27/90] Downloaded ELP 2025
[28/90] Downloaded GEG 2023
[29/90] Downloaded GEG 2024
[30/90] Downloaded GEG 2025
[31/90] Downloaded GRR 2023
[32/90] Downloaded GRR 2024
[33/90] Downloaded GRR 2025
[34/90] Downloaded JAX 2023
[35/90] Downloaded JAX 2024
[36/90] Downloaded JAX 2025
[

In [ ]:
col_names = ['YEAR', 'MONTH', 'DAY', 'HOUR', 'TEMP', 'DEW_POINT', 
             'PRESSURE', 'WIND_DIR', 'WIND_SPEED', 'SKY_CONDITION',
             'PRECIP_1HR', 'PRECIP_6HR']

all_weather = []
raw_dir = 'dataset/weather/raw'

for f in sorted(os.listdir(raw_dir)):
    if not f.endswith('.gz'):
        continue
    airport = f.split('_')[0]
    filepath = os.path.join(raw_dir, f)
    
    try:
        with gzip.open(filepath, 'rt') as gz:
            df = pd.read_fwf(gz, header=None, names=col_names)
        df['AIRPORT'] = airport
        all_weather.append(df)
    except Exception as e:
        print(f'FAILED {f}: {e}')

weather = pd.concat(all_weather, ignore_index=True)

# Clean
weather = weather.replace(-9999, pd.NA)
weather['TEMP'] = weather['TEMP'] / 10
weather['DEW_POINT'] = weather['DEW_POINT'] / 10
weather['PRESSURE'] = weather['PRESSURE'] / 10
weather['WIND_SPEED'] = weather['WIND_SPEED'] / 10
weather['PRECIP_1HR'] = weather['PRECIP_1HR'] / 10
weather['PRECIP_6HR'] = weather['PRECIP_6HR'] / 10

weather['DATETIME'] = pd.to_datetime(
    weather[['YEAR','MONTH','DAY','HOUR']].rename(
        columns={'YEAR':'year','MONTH':'month','DAY':'day','HOUR':'hour'}
    )
)

# Save
weather.to_parquet('dataset/weather/weather_clean.parquet', index=False)

print(f"Total: {len(weather):,} rows")
print(f"Airports: {weather['AIRPORT'].nunique()}")
print(f"Range: {weather['DATETIME'].min()} to {weather['DATETIME'].max()}")

Total: 1,858,098 rows
Airports: 80
Range: 2023-01-01 00:00:00 to 2025-08-27 09:00:00


In [ ]:
airport_stations_next20 = {
    'ALB': '725180-14735', 'AVL': '723150-03812', 'BZN': '726797-24132',
    'COS': '724660-93037', 'DSM': '725460-14933', 'FAT': '723890-93193',
    'GSO': '723170-13723', 'GSP': '723120-03870', 'HPN': '725037-94745',
    'LIT': '723403-13963', 'MSN': '726410-14837', 'MYR': '747910-13717',
    'PNS': '722223-13899', 'PSP': '722868-93138', 'PVD': '725070-14765',
    'PWM': '726060-14764', 'ROC': '725290-14768', 'SYR': '725190-14771',
    'TYS': '723260-13891', 'XNA': '723436-53922',
}

base_url = 'https://www.ncei.noaa.gov/pub/data/noaa/isd-lite'
years = ['2023', '2024', '2025']

total = len(airport_stations_next20) * len(years)
done = 0

for airport, station_id in airport_stations_next20.items():
    for year in years:
        filename = f'{station_id}-{year}.gz'
        url = f'{base_url}/{year}/{filename}'
        save_path = f'dataset/weather/raw/{airport}_{year}.gz'
        
        try:
            urllib.request.urlretrieve(url, save_path)
            done += 1
            print(f'[{done}/{total}] Downloaded {airport} {year}')
        except Exception as e:
            print(f'[FAILED] {airport} {year}: {e}')

print(f'\nDone! {done}/{total} files downloaded.')

[1/60] Downloaded ALB 2023
[2/60] Downloaded ALB 2024
[3/60] Downloaded ALB 2025
[4/60] Downloaded AVL 2023
[5/60] Downloaded AVL 2024
[6/60] Downloaded AVL 2025
[7/60] Downloaded BZN 2023
[8/60] Downloaded BZN 2024
[9/60] Downloaded BZN 2025
[10/60] Downloaded COS 2023
[11/60] Downloaded COS 2024
[12/60] Downloaded COS 2025
[13/60] Downloaded DSM 2023
[14/60] Downloaded DSM 2024
[15/60] Downloaded DSM 2025
[16/60] Downloaded FAT 2023
[17/60] Downloaded FAT 2024
[18/60] Downloaded FAT 2025
[19/60] Downloaded GSO 2023
[20/60] Downloaded GSO 2024
[21/60] Downloaded GSO 2025
[22/60] Downloaded GSP 2023
[23/60] Downloaded GSP 2024
[24/60] Downloaded GSP 2025
[25/60] Downloaded HPN 2023
[26/60] Downloaded HPN 2024
[27/60] Downloaded HPN 2025
[28/60] Downloaded LIT 2023
[29/60] Downloaded LIT 2024
[30/60] Downloaded LIT 2025
[31/60] Downloaded MSN 2023
[32/60] Downloaded MSN 2024
[33/60] Downloaded MSN 2025
[34/60] Downloaded MYR 2023
[35/60] Downloaded MYR 2024
[36/60] Downloaded MYR 2025
[

In [ ]:
col_names = ['YEAR', 'MONTH', 'DAY', 'HOUR', 'TEMP', 'DEW_POINT', 
             'PRESSURE', 'WIND_DIR', 'WIND_SPEED', 'SKY_CONDITION',
             'PRECIP_1HR', 'PRECIP_6HR']

all_weather = []
raw_dir = 'dataset/weather/raw'

for f in sorted(os.listdir(raw_dir)):
    if not f.endswith('.gz'):
        continue
    airport = f.split('_')[0]
    filepath = os.path.join(raw_dir, f)
    
    try:
        with gzip.open(filepath, 'rt') as gz:
            df = pd.read_fwf(gz, header=None, names=col_names)
        df['AIRPORT'] = airport
        all_weather.append(df)
    except Exception as e:
        print(f'FAILED {f}: {e}')

weather = pd.concat(all_weather, ignore_index=True)

weather = weather.replace(-9999, pd.NA)
weather['TEMP'] = weather['TEMP'] / 10
weather['DEW_POINT'] = weather['DEW_POINT'] / 10
weather['PRESSURE'] = weather['PRESSURE'] / 10
weather['WIND_SPEED'] = weather['WIND_SPEED'] / 10
weather['PRECIP_1HR'] = weather['PRECIP_1HR'] / 10
weather['PRECIP_6HR'] = weather['PRECIP_6HR'] / 10

weather['DATETIME'] = pd.to_datetime(
    weather[['YEAR','MONTH','DAY','HOUR']].rename(
        columns={'YEAR':'year','MONTH':'month','DAY':'day','HOUR':'hour'}
    )
)

weather.to_parquet('dataset/weather/weather_clean.parquet', index=False)

print(f"Total: {len(weather):,} rows")
print(f"Airports: {weather['AIRPORT'].nunique()}")
print(f"Range: {weather['DATETIME'].min()} to {weather['DATETIME'].max()}")

Total: 2,322,436 rows
Airports: 100
Range: 2023-01-01 00:00:00 to 2025-08-27 09:00:00
